# 31 · Context engineering basics
Level L3 · Part 4 · core · ~35 min · builds on 05, 26

## Goal
Decide what goes into Scout's context window, in what order, and what to leave out. Trim old tool
results so a research run whose last call carries about 8.5k tokens drops to about 2k on the fake-model run (about 6.3k to 2.6k on Gemini), and check that the
answer stays the same, first with a fake model and then with Gemini.

## The idea
Lesson 05: the window is finite. In an agent,
the window fills mostly with **tool results**, and the loop from lesson 26 re-sends every one of them
on every call. **Context engineering** is choosing, for each call, the smallest set of tokens that
still lets the model do the next step: what goes in, in what order, and what stays out. Two simple
tools: **just-in-time retrieval** (give the model names of things and a tool to read them, not the
things) and **tool-result trimming** (replace results the model has already used with a short note).
Analogy: a desk. You keep the folder you are working on open, and the rest stay labelled on the shelf.

## Picture
```mermaid
flowchart LR
    H["full history<br/>(kept, never edited)"] --> V["view(messages)<br/>trim old tool results"]
    V --> R["request for this step<br/>system · tools · question · recent results"]
    R --> M[model]
    M --> H
    S["shelf: note ids + titles"] -. "read_note(id)<br/>just in time" .-> H
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 26. Without a key the Gemini calls replay from
`../data/llm_cassettes/31_context-engineering.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("31_context-engineering")

LLM mode: replay (31_context-engineering.yaml)


Setup from lesson 26, unchanged: the client, the pinned model and the safe `calculator`.

In [2]:
import ast, json, operator, os, random
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

### 2. A token counter
To engineer the context we must measure it. We count with tiktoken's `o200k_base` (lesson 03), loaded
from the cache in `../data/tiktoken_cache`, so it works offline. It is an estimate: Gemini's tokenizer
differs a little, and lesson 12 showed that bills come from the response's `usage_metadata`. We add 4
tokens per message for the role markers, as real chat formats do.

In [3]:
os.environ["TIKTOKEN_CACHE_DIR"] = "../data/tiktoken_cache"
import tiktoken
ENC = tiktoken.get_encoding("o200k_base")

def tokens(text: str) -> int:
    return len(ENC.encode(text))

def message_tokens(m: dict) -> int:
    body = m.get("text") or m.get("content") or json.dumps(m.get("tool_calls", []))
    return 4 + tokens(body)

print(tokens("A day on Mars lasts about 24 hours and 39 minutes."), "tokens")

14 tokens


### 3. Long notes
Lesson 26's notes were one sentence each. Real notes are long. For this lesson Scout's notes are three
field notebooks from a Mars rover project: daily log lines, generated with a fixed seed so every run is
the same. The fact we need, the length of a Martian day, sits at the end of notebook 3.

In [4]:
rng = random.Random(31)

def rover_log(first_sol: int, n: int) -> str:
    return "\n".join(f"Sol {s}: drove {rng.randint(0, 120)} m, wind {rng.randint(1, 15)} m/s, dust opacity "
                     f"{rng.uniform(0.2, 1.5):.2f}, battery {rng.randint(40, 100)}%, photos {rng.randint(0, 60)}."
                     for s in range(first_sol, first_sol + n))

NOTEBOOKS = {
    1: {"title": "Mars field notebook 1", "text": rover_log(1, 100)},
    2: {"title": "Mars field notebook 2", "text": rover_log(101, 100)},
    3: {"title": "Mars field notebook 3", "text": rover_log(201, 55) +
        "\nReminder for the talk: a day on Mars (a sol) lasts 24 hours and 39 minutes."},
}
for i, nb in NOTEBOOKS.items():
    print(i, nb["title"], "|", tokens(nb["text"]), "tokens |", nb["text"][:60], "...")

1 Mars field notebook 1 | 3000 tokens | Sol 1: drove 1 m, wind 8 m/s, dust opacity 0.35, battery 65% ...
2 Mars field notebook 2 | 3000 tokens | Sol 101: drove 40 m, wind 14 m/s, dust opacity 0.99, battery ...
3 Mars field notebook 3 | 1672 tokens | Sol 201: drove 71 m, wind 9 m/s, dust opacity 1.30, battery  ...


### 4. What goes in: everything up front, or just in time?
The simplest design pastes every notebook into the system prompt. Then every call, even "what is
2+2?", carries all of them. The other design is **just-in-time retrieval**: the model gets
**lightweight identifiers** (id, title, size) from a search tool and reads a notebook only when it
decides it needs it. Compare what each puts in the window before the model has done anything.

In [5]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Use search_notes to find notes and "
          "read_note to read one. The notes are long: read one at a time, and stop reading once you have "
          "what you need. Do every calculation with the calculator tool. Answer in one sentence.")

def search_notes(query: str) -> list[dict]:
    """Return id, title and size of every notebook whose title or text contains `query`."""
    q = query.lower()
    return [{"id": i, "title": nb["title"], "tokens": tokens(nb["text"])}
            for i, nb in NOTEBOOKS.items() if q in (nb["title"] + nb["text"]).lower()]

def read_note(note_id: int) -> str:
    """Return the full text of one notebook."""
    return NOTEBOOKS[note_id]["text"]

everything = SYSTEM + "\n\n" + "\n\n".join(nb["text"] for nb in NOTEBOOKS.values())
print("everything up front:", tokens(everything), "tokens on every call")
print("just in time       :", tokens(SYSTEM) + tokens(json.dumps(search_notes("Mars"))),
      "tokens (system + one search result)")

everything up front: 7730 tokens on every call
just in time       : 128 tokens (system + one search result)


Just in time wins as long as most questions need only part of the notes. It costs extra steps (one
call per read), which is why Claude Code, for example, loads a small project file up front and finds
everything else with search tools. Now the schemas, with `read_note` added.

In [6]:
def schema(name: str, description: str, prop: str, kind: str, example: str) -> dict:
    return {"name": name, "description": description,
            "parameters": {"type": "object", "required": [prop],
                           "properties": {prop: {"type": kind, "description": example}}}}

SCHEMAS = [
    schema("search_notes", "Search the user's notebooks. Returns {id, title, tokens} for every notebook "
           "whose title or text contains the query word. Does not return the text.", "query", "string",
           "one word, e.g. 'Mars'"),
    schema("read_note", "Return the full text of one notebook by id. Notebooks are long.", "note_id",
           "integer", "an id from search_notes, e.g. 2"),
    schema("calculator", "Evaluate arithmetic with numbers, + - * / and brackets, e.g. '24*60+39'.",
           "expression", "string", "e.g. '(3+4)*2'"),
]
TOOLS = {"search_notes": search_notes, "read_note": read_note, "calculator": calculator}

def context_tokens(messages: list[dict]) -> int:
    """Everything sent on one call: system prompt, tool schemas and messages."""
    return tokens(SYSTEM) + tokens(json.dumps(SCHEMAS)) + sum(message_tokens(m) for m in messages)

print("fixed part of every call:", context_tokens([]), "tokens")

fixed part of every call: 292 tokens


### 5. The loop, with a view
Lesson 26's `ScriptedModel` and `check_history`, unchanged, and its `run_agent` with two changes. The
model now gets `view(messages)`, a request built from the history, so the history itself is never
edited. And each tool result remembers its `args` and `step`, and the loop records `sizes`, the tokens
sent on each call.

In [7]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

Now `run_agent`. The lines marked NEW are the only changes from lesson 26.

In [8]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8,
              view=lambda messages: messages) -> dict:
    messages, sizes = [{"role": "user", "text": question}], []
    for step in range(1, max_steps + 1):
        request = view(messages)                              # NEW: what the model sees this step
        sizes.append(context_tokens(request))
        reply = model(request, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "messages": messages, "sizes": sizes}
        for call in calls:
            result = tools[call["name"]](**call["args"])
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "args": call["args"], "step": step, "content": json.dumps(result)})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "messages": messages, "sizes": sizes}

### 6. A research run that fills the window
The fake model plays a careful researcher: search, read notebooks 1, 2 and 3 in turn, calculate, then
answer from the calculator's result.

In [9]:
QUESTION = "How many minutes long is a day on Mars?"
def last_result(messages): return json.loads(messages[-1]["content"])

RESEARCH_SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "Mars"}}]},
    {"tool_calls": [{"name": "read_note", "args": {"note_id": 1}}]},
    {"tool_calls": [{"name": "read_note", "args": {"note_id": 2}}]},
    {"tool_calls": [{"name": "read_note", "args": {"note_id": 3}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda messages: {"text": f"A day on Mars lasts {last_result(messages):,.0f} minutes."},
]
full = run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS)
print(full["stop"], "after", full["steps"], "steps:", full["answer"])
print("tokens sent per call:", full["sizes"], "| total:", sum(full["sizes"]))

end_of_turn after 6 steps: A day on Mars lasts 1,479 minutes.
tokens sent per call: [306, 410, 3545, 6680, 8443, 8482] | total: 27866


The last call sends about 8.5k tokens to say one number. Where do they go? Count by role.

In [10]:
def breakdown(messages: list[dict]) -> dict:
    parts = {"system + tools": context_tokens([])}
    for m in messages:
        parts[m["role"]] = parts.get(m["role"], 0) + message_tokens(m)
    return parts

print(breakdown(full["messages"][:-1]))          # what the final call received

{'system + tools': 292, 'user': 14, 'assistant': 156, 'tool': 8020}


Tool results are almost all of it. Notebooks 1 and 2 did not contain the answer, and by the last step
the model has already turned notebook 3 into `24*60+39`. Yet all three are re-sent on every call. And
the total over the run grows roughly with the square of the steps, because each call re-sends
everything before it.

### 7. Trim old tool results
`trim_old_results` builds the view. Results from the last `keep_steps` tool steps stay. Older ones are
replaced by a short note saying what was removed and how to get it back. The calls stay, so every
result still answers a call (lesson 22). The newest step is always kept: the model has not seen it yet.

In [11]:
def trim_old_results(messages: list[dict], keep_steps: int = 2) -> list[dict]:
    steps = sorted({m["step"] for m in messages if m["role"] == "tool"})
    recent = set(steps[len(steps) - keep_steps:])
    view = []
    for m in messages:
        if m["role"] == "tool" and m["step"] not in recent:
            note = (f"{m['name']}({json.dumps(m['args'])}) returned {tokens(m['content'])} tokens, "
                    "cleared to save space. Call it again if you need them.")
            m = {**m, "content": json.dumps({"cleared": note})}     # a copy: history is untouched
        view.append(m)
    return view

for m in trim_old_results(full["messages"][:-1])[1:]:
    print(f"{m['role']:9} {message_tokens(m):5} tokens | {str(m.get('content', m.get('tool_calls')))[:75]}")

assistant    30 tokens | [{'name': 'search_notes', 'args': {'query': 'Mars'}, 'id': 'call_1'}]
tool         38 tokens | {"cleared": "search_notes({\"query\": \"Mars\"}) returned 70 tokens, cleare
assistant    31 tokens | [{'name': 'read_note', 'args': {'note_id': 1}, 'id': 'call_2'}]
tool         39 tokens | {"cleared": "read_note({\"note_id\": 1}) returned 3100 tokens, cleared to s
assistant    31 tokens | [{'name': 'read_note', 'args': {'note_id': 2}, 'id': 'call_3'}]
tool         39 tokens | {"cleared": "read_note({\"note_id\": 2}) returned 3100 tokens, cleared to s
assistant    31 tokens | [{'name': 'read_note', 'args': {'note_id': 3}, 'id': 'call_4'}]
tool       1732 tokens | "Sol 201: drove 71 m, wind 9 m/s, dust opacity 1.30, battery 49%, photos 41
assistant    33 tokens | [{'name': 'calculator', 'args': {'expression': '24*60+39'}, 'id': 'call_5'}
tool          6 tokens | 1479


Now the same run with the trimmed view. Same script, same question.

In [12]:
trimmed = run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS, view=trim_old_results)
print(trimmed["stop"], "after", trimmed["steps"], "steps:", trimmed["answer"])
print("tokens sent per call:", trimmed["sizes"], "| total:", sum(trimmed["sizes"]))
print(f"last call: {full['sizes'][-1]} -> {trimmed['sizes'][-1]} tokens; "
      f"same answer: {trimmed['answer'] == full['answer']}")
print("history still complete:", len(trimmed["messages"]) == len(full["messages"]))

end_of_turn after 6 steps: A day on Mars lasts 1,479 minutes.
tokens sent per call: [306, 410, 3545, 6644, 5342, 2316] | total: 18563
last call: 8482 -> 2316 tokens; same answer: True
history still complete: True


On the fake-model run the last call shrinks from about 8.5k to about 2k tokens and the answer is identical. The stored history
still holds every notebook, so logs and later debugging lose nothing (lesson 49).

### 8. In what order
Order matters for two reasons. Providers cache a request's **stable prefix**: if call 2 starts with
the same tokens as call 1, that part can be reused cheaply (lesson 41). So put what never changes first
(system prompt, tool schemas), then the history, and anything that changes every call last. Here is
what a "helpful" timestamp at the top of the system prompt does to the shared prefix.

In [13]:
def shared_prefix_tokens(a: str, b: str) -> int:
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return tokens(a[:n])

def request_text(messages: list[dict], header: str = "") -> str:
    return header + SYSTEM + json.dumps(SCHEMAS) + json.dumps([m.get("text") or m.get("content") or
                                                                m.get("tool_calls") for m in messages])

call4, call5 = full["messages"][:6], full["messages"][:8]          # what calls 4 and 5 received
print("stable first        :", shared_prefix_tokens(request_text(call4), request_text(call5)), "tokens shared")
print("timestamp at the top:", shared_prefix_tokens(request_text(call4, "Time: 10:41:07. "),
                                                  request_text(call5, "Time: 10:41:09. ")), "tokens shared")

stable first        : 3651 tokens shared
timestamp at the top: 8 tokens shared


The second reason is quality. Anthropic's long-context guide: "Queries at the end can improve response
quality by up to 30 percent in tests." Our loop already does this: the question comes after the system prompt, and the latest
result, the one the model needs next, comes last.

Trimming also changes the request in the middle, so it breaks the cached prefix from that point. That
is why trimming in big steps (many tokens at once) beats trimming a little every call.

### 9. The same experiment with Gemini
Lesson 26's Gemini wrapper, with one addition: it stores `usage_metadata.prompt_token_count`, the real
input tokens of each call (lesson 12). First `to_contents`, unchanged from lesson 26.

In [14]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                        # unchanged: keeps the thought signature
        else:                                                # a tool result
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response={"result": json.loads(m["content"])}))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)              # same step: same user turn
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

Then `GeminiModel`, with the one NEW line that keeps the real prompt token count of each call.

In [15]:
class GeminiModel:
    def __init__(self):
        self.prompt_tokens = []                                     # NEW: real input tokens per call

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=t["name"], description=t["description"],
                                           parameters_json_schema=t["parameters"]) for t in tools]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=[types.Tool(function_declarations=decls)],
                                          automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        self.prompt_tokens.append(resp.usage_metadata.prompt_token_count)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

Run the question twice: full history, then trimmed. We print the tools Gemini chose, the real prompt
tokens of each call, and the answer.

In [16]:
gemini_runs = {}
for name, view in [("full", lambda messages: messages), ("trimmed", trim_old_results)]:
    gm = GeminiModel()
    run = run_agent(QUESTION, gm, TOOLS, SCHEMAS, view=view)
    gemini_runs[name] = {"run": run, "prompt_tokens": gm.prompt_tokens}
    print(f"{name}: {run['stop']} after {run['steps']} steps")
    print("  tools:", [f"{c['name']}({c['args']})" for m in run["messages"] for c in m.get("tool_calls", [])])
    print("  prompt tokens per call:", gm.prompt_tokens, "| total:", sum(gm.prompt_tokens))
    print("  answer:", run["answer"])

full: end_of_turn after 7 steps
  tools: ["search_notes({'query': 'Mars'})", "read_note({'note_id': 1})", "search_notes({'query': 'sol'})", "search_notes({'query': 'hours'})", "read_note({'note_id': 3})", "calculator({'expression': '24*60+39'})"]
  prompt tokens per call: [236, 292, 4048, 4104, 4134, 6285, 6309] | total: 25408
  answer: A day on Mars is 1479 minutes long.
trimmed: end_of_turn after 7 steps
  tools: ["search_notes({'query': 'Mars'})", "read_note({'note_id': 1})", "search_notes({'query': 'sol'})", "search_notes({'query': 'hours'})", "read_note({'note_id': 3})", "calculator({'expression': '24 * 60 + 39'})"]
  prompt tokens per call: [236, 292, 4048, 4095, 422, 2564, 2607] | total: 14264
  answer: A Martian day, or sol, is 1,479 minutes long.


In the recorded run Gemini did not read all three notebooks. It read notebook 1, found nothing, then
searched for "sol" and "hours" and read only notebook 3:
just-in-time retrieval chosen by the model. Both runs took the same 7 steps and gave the same number,
1,479 minutes, worded differently. With trimming, the last call fell from 6,309 to 2,607 real prompt
tokens, and the run total from 25,408 to 14,264. Gemini's counts differ from our tiktoken estimate, but
the shape is the same. One run is not proof; lesson 48 measures this over many questions.

## At work
- **Anthropic's API trims tool results for you.** Context editing (beta, `clear_tool_uses_20250919`)
  keeps the last 3 tool uses past 100,000 input tokens and replaces older results with a placeholder.
  Your own history stays whole. Beyond trimming, history gets summarised → lesson 32.
- **Clearing breaks the cache** from that point, so teams clear a minimum amount (`clear_at_least`),
  not every call → lesson 41.
- **Coding agents work just in time.** Claude Code loads a small `CLAUDE.md` up front and finds the
  rest with glob and grep, keeping paths, not file contents, in context → lesson 35.
- **"Same answer" is measured.** A context change ships only after an eval set shows answers did not
  get worse → lessons 48 and 49.

## What just happened
- We counted tokens per call and by role: tool results were almost all of Scout's context.
- **Just-in-time retrieval**: search returns ids, titles and sizes; `read_note` loads one notebook only
  when the model asks. Far fewer tokens than pasting every notebook into the prompt.
- `run_agent` now sends `view(messages)`, so we can shape each request without editing the history.
- **Tool-result trimming**: `trim_old_results` replaces results older than the last 2 tool steps with a
  note saying what was cleared. The last call went from about 8.5k to about 2k tokens, same answer.
- Order: stable parts first keep a long shared prefix; changing parts go last, the question after the
  documents.

## Common mistakes
**1. Using lesson 05's sliding window on an agent history.** Keeping only the newest messages (here
the last 5) cuts a tool call but keeps its result, and drops the question too. The fake rejects the
orphan result; real APIs reject it too.

In [17]:
def sliding_window(messages: list[dict], keep: int = 5) -> list[dict]:
    return messages[-keep:]                       # newest messages only, like lesson 05

run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS, view=sliding_window)

ValueError: tool result for unknown call id 'call_1'

Fix: keep every message and shrink only the content of old results, as `trim_old_results` does.

In [18]:
print(run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS, view=trim_old_results)["answer"])

A day on Mars lasts 1,479 minutes.


**2. Clearing the newest result too.** With `keep_steps=0` the model gets a placeholder instead of the
number it just asked for. Here the answer step tries to format it as a number and fails; a real model
would call the tool again or guess.

In [19]:
run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS,
          view=lambda messages: trim_old_results(messages, keep_steps=0))

TypeError: unsupported format string passed to dict.__format__

Fix: always keep at least the latest tool step (`keep_steps >= 1`); 2 is a safe start.

In [20]:
print(run_agent(QUESTION, ScriptedModel(RESEARCH_SCRIPT), TOOLS, SCHEMAS,
                view=lambda messages: trim_old_results(messages, keep_steps=1))["sizes"])

[306, 410, 3509, 3579, 2277, 623]


## Try it
Anthropic's clearing has `exclude_tools`: results of those tools are never cleared. Write
`trim_except(messages, keep_steps=2, keep_tools={"search_notes"})` that works like `trim_old_results`
but never clears results of tools in `keep_tools`. Run the fake research script with it, print the
tokens per call, and check the answer is still the same as `full["answer"]`.

In [21]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [22]:
llm_replay.stop()

## Key terms
- **context engineering** — choosing, for each model call, the smallest set of tokens that lets the model do the next step: what goes in, in what order, and what stays out.
- **just-in-time retrieval** — giving the model identifiers and a tool to load content when it needs it, instead of putting all the content in the prompt up front.
- **lightweight identifier** — a short reference to content (an id, title, file path or query) that stays in context in place of the content itself.
- **tool-result trimming** — replacing old tool results in the request with a short placeholder, while keeping the tool calls and the full stored history.
- **stable prefix** — the start of a request that stays identical from call to call (system prompt, tool schemas, early history); keeping it stable lets providers reuse it.